In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install tensorflow scikit-learn pandas matplotlib seaborn requests

In [ ]:
import os, json, time, math, argparse, datetime, hashlib
import numpy as np
import pandas as pd
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

class CONFIG:
    N_ARTIFACTS   = 300   
    N_DAYS        = 365 * 4    
    SEQ_LEN       = 60        
    STRIDE        = 4    
    VOXEL_RES     = 20    
    GRU_UNITS     = 50
    ATTN_HEADS    = 4
    DENSE_WIDTH   = 128
    BATCH_SIZE    = 128         
    EPOCHS        = 80          
    LR            = 3e-4
    PATIENCE      = 12
    MAX_HOURS     = 6.0         
    VAL_SPLIT     = 0.15
    TEST_SPLIT    = 0.15
    HAZARD_FRACTION = 0.05      
    DRIVE_ROOT    = "/content/drive/MyDrive/artifact_risk"
    LOCAL_ROOT    = "./artifact_risk_out"
    CLIMATE_CSV   = "climate.csv"   
    USE_REAL_CLIMATE = True         
    SITE_LAT      = 50.28           
    SITE_LON      = 57.17
    CLIMATE_START = "2018-01-01"

    MATERIALS = ["bronze", "iron", "stone", "ceramic", "wood", "glass", "textile", "bone"]
    STORAGES  = ["indoor_climate_controlled", "indoor_uncontrolled",
                 "outdoor_sheltered", "outdoor_exposed", "underground_vault"]
    FACTORS   = ["none", "flood", "seismic", "thermal_shock", "corrosion"]  #softmax classes

    ENV_FEATURES = ["air_temp", "humidity", "water_temp", "water_level", "rain",
                    "vibration", "gas_ppm", "pressure_hpa", "wind_ms"]  #9features
    META_NUM = ["age", "latitude", "longitude", "height_cm", "weight_kg",
                "existing_cracks", "largest_crack_mm", "corrosion_percent",
                "surface_loss_percent"]


def resolve_root():
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(CONFIG.DRIVE_ROOT, exist_ok=True)
        return CONFIG.DRIVE_ROOT
    os.makedirs(CONFIG.LOCAL_ROOT, exist_ok=True)
    return CONFIG.LOCAL_ROOT
def _load_climate_csv(path):
    raw = pd.read_csv(path)
    raw.columns = [c.strip().lower() for c in raw.columns]
    date_col = next((c for c in ["date", "time", "datetime", "fecha", "ds", "day"]
                     if c in raw.columns), None)
    if date_col is None:
        raise ValueError("CSV needs a date column (date/time/datetime/...).")
    raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
    raw = raw.dropna(subset=[date_col]).sort_values(date_col)
    raw = raw.drop_duplicates(subset=[date_col]).set_index(date_col)

    aliases = {
        "air_temp":     ["air_temp", "temperature_2m_mean", "t_air", "temp", "tavg", "t"],
        "humidity":     ["humidity", "relative_humidity_2m_mean", "rh", "hum"],
        "water_temp":   ["water_temp", "t_water", "wtemp"],
        "water_level":  ["water_level", "level", "wlevel", "stage"],
        "rain":         ["rain", "precipitation_sum", "precip", "prcp", "rainfall"],
        "vibration":    ["vibration", "seismic", "quake"],
        "gas_ppm":      ["gas_ppm", "gas", "co2", "ppm"],
        "pressure_hpa": ["pressure_hpa", "surface_pressure_mean", "pressure", "pres", "slp"],
        "wind_ms":      ["wind_ms", "wind_speed_10m_max", "wind", "wspd", "wind_speed"],
    }
    df = pd.DataFrame(index=raw.index)
    have = {}
    for feat, names in aliases.items():
        col = next((n for n in names if n in raw.columns), None)
        if col is not None:
            df[feat] = pd.to_numeric(raw[col], errors="coerce")
            have[feat] = True
    df = df.apply(lambda s: s.interpolate().ffill().bfill())
    if "wind_ms" in df and df["wind_ms"].mean() > 60:
        df["wind_ms"] = df["wind_ms"] / 3.6
    #name chng
    if "rain" in df: 
        df["precip"] = df["rain"]
    df = _derive_artifact_channels(df)
    print(f"[climate] REAL CSV loaded: {len(df)} days, "
          f"supplied features = {sorted(have)}")
    return df


def load_real_climate(root):
    cache = os.path.join(root, "climate_real.pkl")
    if os.path.exists(cache):
        print(f"[climate] using cached record: {cache}")
        return pd.read_pickle(cache)

    csv_path = os.path.join(root, CONFIG.CLIMATE_CSV)
    if os.path.exists(csv_path):
        df = _load_climate_csv(csv_path)
        df.to_pickle(cache)
        return df
    start = CONFIG.CLIMATE_START
    end = (pd.Timestamp(start) + pd.Timedelta(days=CONFIG.N_DAYS - 1)).strftime("%Y-%m-%d")
    if CONFIG.USE_REAL_CLIMATE:
        try:
            import requests
            url = ("https://archive-api.open-meteo.com/v1/archive"
                   f"?latitude={CONFIG.SITE_LAT}&longitude={CONFIG.SITE_LON}"
                   f"&start_date={start}&end_date={end}"
                   "&daily=temperature_2m_mean,relative_humidity_2m_mean,"
                   "precipitation_sum,surface_pressure_mean,wind_speed_10m_max"
                   "&timezone=UTC")
            print("[climate] no CSV found; fetching real Open-Meteo archive ...")
            r = requests.get(url, timeout=60); r.raise_for_status()
            d = r.json()["daily"]
            df = pd.DataFrame({
                "date":        pd.to_datetime(d["time"]),
                "air_temp":    np.array(d["temperature_2m_mean"], float),
                "humidity":    np.array(d["relative_humidity_2m_mean"], float),
                "precip":      np.array(d["precipitation_sum"], float),
                "pressure_hpa":np.array(d["surface_pressure_mean"], float),
                "wind_ms":     np.array(d["wind_speed_10m_max"], float) / 3.6,
            }).set_index("date").interpolate().ffill().bfill()
            df = _derive_artifact_channels(df)
            df.to_pickle(cache)
            print(f"[climate] REAL archive loaded: {len(df)} days -> {cache}")
            return df
        except Exception as e:
            print(f"[climate] real fetch unavailable ({type(e).__name__}: {e}). "
                  f"Falling back to deterministic generator.")

    df = _synthesize_climate(start, CONFIG.N_DAYS)
    df.to_pickle(cache)
    return df

def _derive_artifact_channels(df):
    df = df.copy()
    n = len(df)
    yr = 2 * np.pi * np.arange(n) / 365.25
    if "air_temp" not in df:     df["air_temp"] = 8 + 14 * np.sin(yr - 1.2)
    if "humidity" not in df:     df["humidity"] = (60 + 20 * np.sin(yr + 0.5)).clip(10, 100)
    if "precip" not in df:       df["precip"] = np.clip(3 * (np.sin(yr * 3) + 1) * ((np.arange(n) % 9) < 2), 0, None)
    if "pressure_hpa" not in df: df["pressure_hpa"] = 1013 + 6 * np.sin(yr * 2)
    if "wind_ms" not in df:      df["wind_ms"] = (3 + 2 * np.abs(np.sin(yr * 5))).clip(0, None)

    if "rain" not in df:        df["rain"] = (df["precip"] > 1.0).astype(int)
    else:                       df["rain"] = (df["rain"] > 0).astype(int)
    if "water_temp" not in df:  df["water_temp"] = df["air_temp"].rolling(7, min_periods=1).mean() * 0.6 + 6.0
    if "water_level" not in df:
        df["water_level"] = (40 + 8 * df["precip"].rolling(15, min_periods=1).sum()
                             .clip(upper=30)).clip(lower=0)
    if "vibration" not in df:
        idx = np.arange(n)
        df["vibration"] = (((idx * 1103515245 + 12345) % 211) == 0).astype(int)
    else:
        df["vibration"] = (df["vibration"] > 0).astype(int)
    if "gas_ppm" not in df:     df["gas_ppm"] = (350 + 4 * df["humidity"]).clip(lower=350)
    return df[CONFIG.ENV_FEATURES]


def _synthesize_climate(start, n):
    print("[climate] *** OFFLINE FALLBACK: deterministic synthetic climate ***")
    t = np.arange(n)
    yr = 2 * np.pi * t / 365.25
    air = 8 + 14 * np.sin(yr - 1.2)                       #seasonal
    hum = (60 + 20 * np.sin(yr + 0.5)).clip(10, 100)
    pre = np.clip(3 * (np.sin(yr * 3) + 1) * ((t % 9) < 2), 0, None)
    prs = 1013 + 6 * np.sin(yr * 2) 
    wnd = (3 + 2 * np.abs(np.sin(yr * 5))).clip(0, None)
    df = pd.DataFrame({
        "air_temp": air, "humidity": hum, "precip": pre,
        "pressure_hpa": prs, "wind_ms": wnd,
    }, index=pd.date_range(start, periods=n, freq="D"))
    return _derive_artifact_channels(df)

def load_artifacts(root):
    path = os.path.join(root, "artifacts.json")
    if os.path.exists(path):
        print(f"[meta] using REAL artifacts.json ({path})")
        with open(path) as f:
            recs = json.load(f)
        return pd.DataFrame(recs)

    print("[meta] artifacts.json not found -> deterministic catalogue "
          "(replace with your real inspection records).")
    rows = []
    for i in range(CONFIG.N_ARTIFACTS):
        h = int(hashlib.md5(f"artifact_{i}".encode()).hexdigest(), 16)
        mat = CONFIG.MATERIALS[h % len(CONFIG.MATERIALS)]
        sto = CONFIG.STORAGES[(h // 7) % len(CONFIG.STORAGES)]
        rows.append({
            "artifact_name": f"AR-{i:05d}",
            "material": mat,
            "storage_type": sto,
            "age": 50 + (h % 3000),
            "latitude": CONFIG.SITE_LAT + ((h % 200) - 100) / 1000.0,
            "longitude": CONFIG.SITE_LON + ((h // 13 % 200) - 100) / 1000.0,
            "height_cm": 10 + (h % 250),
            "weight_kg": 0.5 + (h % 800) / 10.0,
            "existing_cracks": h % 12,
            "largest_crack_mm": (h % 50) / 2.0,
            "corrosion_percent": (h % 90),
            "surface_loss_percent": (h // 3 % 70),
        })
    return pd.DataFrame(rows)

 
def voxelize(meta_row, res):
    """Build a deterministic res^3 voxel occupancy grid representing the artifact
    as a tapered solid with carved cracks/corrosion voids. Replace with a real
    mesh->voxel step (e.g. trimesh.voxelized) when you have scan data."""
    g = np.zeros((res, res, res), np.float32)
    cx = cy = res / 2.0
    #radius
    rad = 0.25 + 0.20 * (meta_row["weight_kg"] % 50) / 50.0
    htop = int(res * (0.55 + 0.40 * (meta_row["height_cm"] % 200) / 200.0))
    zz, yy, xx = np.meshgrid(np.arange(res), np.arange(res), np.arange(res), indexing="ij")
    taper = 1.0 - 0.4 * (zz / res)
    rr = np.sqrt(((xx - cx) / res) ** 2 + ((yy - cy) / res) ** 2)
    solid = (rr < rad * taper) & (zz < htop)
    g[solid] = 1.0
    ncr = int(meta_row["existing_cracks"])
    for c in range(ncr):
        ang = (c / max(ncr, 1)) * math.pi
        plane = np.abs((xx - cx) * math.cos(ang) + (yy - cy) * math.sin(ang)) < 0.8
        depth = zz < int(htop * (0.3 + 0.5 * meta_row["largest_crack_mm"] / 25.0))
        g[plane & depth] = 0.0
    if meta_row["corrosion_percent"] > 40:
        shell = solid & (rr > rad * taper - 0.06)
        g[shell] *= 0.3
    return g[..., None] 
def physical_labels(env_window, meta_row):
 
    e = pd.DataFrame(env_window, columns=CONFIG.ENV_FEATURES)
    flood    = (e["rain"].mean() * e["water_level"].mean() / 80.0)
    seismic  = e["vibration"].mean() * 6.0
    thermal  = np.abs(np.diff(e["water_temp"])).mean() / 5.0
    humidity = e["humidity"].mean() / 100.0
    corrode  = (meta_row["corrosion_percent"] / 100.0) * (0.4 + humidity)

    age_f    = min(meta_row["age"] / 3000.0, 1.0)
    crack_f  = min((meta_row["existing_cracks"] + meta_row["largest_crack_mm"]) / 40.0, 1.0)

    degradation = np.tanh(0.6 * corrode + 0.3 * thermal + 0.3 * age_f + 0.2 * humidity)
    collapse    = np.tanh(0.7 * crack_f + 0.5 * flood + 0.4 * seismic + 0.2 * age_f)
    reg = np.array([degradation, collapse], np.float32).clip(0, 1)

    scores = {"flood": flood, "seismic": seismic, "thermal_shock": thermal,
              "corrosion": corrode}
    top, val = max(scores.items(), key=lambda kv: kv[1])
    cls = CONFIG.FACTORS.index(top) if val > 0.35 else 0 
    return reg, cls 
def build_dataset(root):
    climate = load_real_climate(root)
    arts = load_artifacts(root)
    cfg = CONFIG

    env_raw = climate[cfg.ENV_FEATURES].to_numpy(np.float32)
    env_scaled = MinMaxScaler().fit_transform(env_raw).astype(np.float32)
    meta_scaler = MinMaxScaler().fit(arts[cfg.META_NUM].to_numpy(np.float32))
    meta_bank = meta_scaler.transform(arts[cfg.META_NUM].to_numpy(np.float32)).astype(np.float32)

    print(f"[voxel] building {len(arts)} voxel grids @ {cfg.VOXEL_RES}^3 (once each) ...")
    voxel_bank = np.stack([voxelize(arts.iloc[i], cfg.VOXEL_RES)
                           for i in range(len(arts))]).astype(np.float32)

    starts = list(range(0, len(climate) - cfg.SEQ_LEN, cfg.STRIDE))
    env_windows = np.stack([env_scaled[s:s + cfg.SEQ_LEN] for s in starts]).astype(np.float32)
    env_raw_win = [env_raw[s:s + cfg.SEQ_LEN] for s in starts] 
    vib_i = cfg.ENV_FEATURES.index("vibration")
    rain_i = cfg.ENV_FEATURES.index("rain")
    wl_i = cfg.ENV_FEATURES.index("water_level")
    wt_i = cfg.ENV_FEATURES.index("water_temp")
    flood = np.array([w[:, rain_i].mean() * w[:, wl_i].max() for w in env_raw_win])
    seism = np.array([w[:, vib_i].sum() for w in env_raw_win])
    therm = np.array([max(0.0, -np.diff(w[:, wt_i]).min()) if len(w) > 1 else 0.0
                      for w in env_raw_win])
    def _z(a):
        sd = a.std()
        return (a - a.mean()) / sd if sd > 0 else a * 0.0
    haz_score = _z(flood) + _z(seism) + _z(therm)
    thr = np.quantile(haz_score, 1.0 - cfg.HAZARD_FRACTION)
    win_hazard = (haz_score >= thr).astype(np.int32)

    n_art, n_win = len(arts), len(starts)
    print(f"[assemble] {n_art} artifacts x {n_win} windows = {n_art * n_win} samples "
          f"(banks: env {env_windows.nbytes/1e6:.1f}MB, voxel {voxel_bank.nbytes/1e6:.1f}MB)")

    win_id, art_id, y_reg, y_cls, hazard = [], [], [], [], []
    for ai in range(n_art):
        m = arts.iloc[ai]
        for wi in range(n_win):
            reg, cls = physical_labels(env_raw_win[wi], m)
            win_id.append(wi); art_id.append(ai)
            y_reg.append(reg); y_cls.append(cls); hazard.append(win_hazard[wi])

    samples = dict(
        win_id=np.asarray(win_id, np.int32),
        art_id=np.asarray(art_id, np.int32),
        dup=np.zeros(len(win_id), np.int32),
        uid=np.arange(len(win_id), dtype=np.int32),
        y_reg=np.asarray(y_reg, np.float32),
        y_cls=np.asarray(y_cls, np.int32),
        hazard=np.asarray(hazard, np.int32),
    )
    banks = dict(env_windows=env_windows, voxel=voxel_bank, meta_num=meta_bank,
                 material=arts["material"].to_numpy().astype(str),
                 storage=arts["storage_type"].to_numpy().astype(str))
    print(f"[assemble] total samples: {len(samples['win_id'])}, "
          f"raw hazard share: {samples['hazard'].mean()*100:.2f}%")
    return banks, samples, arts
 
def correct_imbalance(samples, target_ratio=0.20, seed=SEED):
    rng = np.random.default_rng(seed)
    hz = samples["hazard"]
    haz_idx = np.where(hz == 1)[0]
    n_all, n_haz = len(hz), len(haz_idx)
    if n_haz == 0:
        print("[imbalance] no hazard samples; skipping."); return samples
    k = max(int(np.ceil((target_ratio * n_all - n_haz) / (1 - target_ratio))), 0)
    if k == 0:
        print(f"[imbalance] hazard share already >= {target_ratio:.0%}; no oversample.")
        return samples
    dup = rng.choice(haz_idx, size=k, replace=True)

    out = {}
    for key, arr in samples.items():
        out[key] = np.concatenate([arr, arr[dup]], axis=0)
    out["dup"] = np.concatenate([np.zeros(n_all, np.int32), np.ones(k, np.int32)])
    out["uid"] = np.arange(n_all + k, dtype=np.int32)  # unique noise seed per row
    final = out["hazard"].mean()
    print(f"[imbalance] +{k} hazard duplicates -> hazard share = {final*100:.4f}% "
          f"(target {target_ratio*100:.0f}%)")
    perm = rng.permutation(len(out["hazard"]))
    return {kk: vv[perm] for kk, vv in out.items()}

 
def build_model(n_classes):
    cfg = CONFIG 
    env_in = layers.Input((cfg.SEQ_LEN, len(cfg.ENV_FEATURES)), name="env")
    x = layers.GRU(cfg.GRU_UNITS, return_sequences=True)(env_in)
    x = layers.Dropout(0.2)(x)
    attn = layers.MultiHeadAttention(num_heads=cfg.ATTN_HEADS,
                                     key_dim=cfg.GRU_UNITS)(x, x)
    x = layers.Add()([x, attn])                    
    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    env_out = layers.Dense(64, activation="relu")(x) 
    num_in = layers.Input((len(cfg.META_NUM),), name="meta_num")
    mat_in = layers.Input((1,), dtype=tf.string, name="material")
    sto_in = layers.Input((1,), dtype=tf.string, name="storage")
    mat_lk = layers.StringLookup(vocabulary=cfg.MATERIALS)
    sto_lk = layers.StringLookup(vocabulary=cfg.STORAGES)
    mat_e = layers.Flatten()(layers.Embedding(mat_lk.vocabulary_size(), 8)(mat_lk(mat_in)))
    sto_e = layers.Flatten()(layers.Embedding(sto_lk.vocabulary_size(), 8)(sto_lk(sto_in)))
    m = layers.Concatenate()([num_in, mat_e, sto_e])
    m = layers.Dense(64, activation="relu")(m)
    meta_out = layers.Dense(32, activation="relu")(m)
 
    vox_in = layers.Input((cfg.VOXEL_RES, cfg.VOXEL_RES, cfg.VOXEL_RES, 1), name="voxel")
    v = layers.Conv3D(16, 3, padding="same", activation="relu")(vox_in)
    v = layers.MaxPooling3D(2)(v)
    v = layers.Conv3D(32, 3, padding="same", activation="relu")(v)
    v = layers.MaxPooling3D(2)(v)
    v = layers.Flatten()(v)
    vox_out = layers.Dense(64, activation="relu")(v)
 
    fused = layers.Concatenate()([env_out, meta_out, vox_out])
    h = layers.Dense(cfg.DENSE_WIDTH, activation="relu")(fused)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(64, activation="relu")(h)
    reg = layers.Dense(2, activation="sigmoid", name="risk")(h)        # %/100
    cls = layers.Dense(n_classes, activation="softmax", name="factor")(h)

    model = Model(
        inputs={"env": env_in, "meta_num": num_in, "material": mat_in,
                "storage": sto_in, "voxel": vox_in},
        outputs={"risk": reg, "factor": cls},
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg.LR),
        loss={"risk": "mse", "factor": "sparse_categorical_crossentropy"},
        loss_weights={"risk": 1.0, "factor": 0.5},
        metrics={
            "risk": [tf.keras.metrics.MeanAbsoluteError(name="mae"),
                     tf.keras.metrics.MeanAbsolutePercentageError(name="mape")],
            "factor": [tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
        },
    )
    return model
 
class TimeBudgetStop(callbacks.Callback):
    def __init__(self, max_hours):
        super().__init__(); self.max_s = max_hours * 3600; self.t0 = None
    def on_train_begin(self, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        el = time.time() - self.t0
        print(f"   [time] elapsed {el/3600:.2f} h / cap {self.max_s/3600:.1f} h")
        if el >= self.max_s:
            print("   [time] wall-clock cap reached -> stopping.")
            self.model.stop_training = True


def make_callbacks(root):
    ck = os.path.join(root, "checkpoints"); os.makedirs(ck, exist_ok=True)
    return [ 
        callbacks.ModelCheckpoint(os.path.join(ck, "epoch_{epoch:03d}.weights.h5"),
                                  save_weights_only=True, save_freq="epoch", verbose=0),
 
        callbacks.ModelCheckpoint(os.path.join(ck, "best.weights.h5"),
                                  save_weights_only=True, monitor="val_loss",
                                  save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor="val_loss", patience=CONFIG.PATIENCE,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=max(4, CONFIG.PATIENCE // 3), verbose=1),
        callbacks.CSVLogger(os.path.join(root, "training_log.csv"), append=False),
        callbacks.TensorBoard(os.path.join(root, "tb")),
        TimeBudgetStop(CONFIG.MAX_HOURS),
    ]

 
def make_tfds(idx, banks, samples, training, noise_std=0.01):
    env_bank = tf.constant(banks["env_windows"])
    vox_bank = tf.constant(banks["voxel"])
    meta_bank = tf.constant(banks["meta_num"])
    mat_bank = tf.constant(banks["material"])
    sto_bank = tf.constant(banks["storage"])

    cols = {k: samples[k][idx] for k in ("win_id", "art_id", "dup", "uid", "y_cls")}
    cols["y_reg"] = samples["y_reg"][idx]
    ds = tf.data.Dataset.from_tensor_slices(cols)

    def _map(r):
        env = tf.gather(env_bank, r["win_id"])
        vox = tf.gather(vox_bank, r["art_id"])
        mn = tf.gather(meta_bank, r["art_id"])
        d = tf.cast(r["dup"], tf.float32)
        uid = tf.cast(r["uid"], tf.int32) 
        env *= 1.0 + d * tf.random.stateless_normal(tf.shape(env), [SEED + 1, uid], stddev=noise_std)
        vox *= 1.0 + d * tf.random.stateless_normal(tf.shape(vox), [SEED + 2, uid], stddev=noise_std)
        mn *= 1.0 + d * tf.random.stateless_normal(tf.shape(mn), [SEED + 3, uid], stddev=noise_std)
        inputs = {
            "env": env, "meta_num": mn, "voxel": vox,
            "material": tf.reshape(tf.gather(mat_bank, r["art_id"]), (1,)),
            "storage": tf.reshape(tf.gather(sto_bank, r["art_id"]), (1,)),
        }
        targets = {"risk": r["y_reg"], "factor": r["y_cls"]}
        return inputs, targets

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(min(len(idx), 8192), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(CONFIG.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
 
def dashboard(history, eval_arrays, root):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    try:
        import seaborn as sns; sns.set_theme(style="whitegrid")
    except Exception:
        pass
    h = history.history
    y_true, y_pred, c_true, c_pred = eval_arrays
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))

    ax[0, 0].plot(h.get("loss", []), label="train")
    ax[0, 0].plot(h.get("val_loss", []), label="val")
    ax[0, 0].set_title("Total loss"); ax[0, 0].legend()

    if "risk_mae" in h:
        ax[0, 1].plot(h["risk_mae"], label="train MAE")
        ax[0, 1].plot(h.get("val_risk_mae", []), label="val MAE")
    ax[0, 1].set_title("Regression MAE"); ax[0, 1].legend()

    if "factor_acc" in h:
        ax[0, 2].plot(h["factor_acc"], label="train acc")
        ax[0, 2].plot(h.get("val_factor_acc", []), label="val acc")
    ax[0, 2].set_title("Classification accuracy"); ax[0, 2].legend()

    ax[1, 0].scatter(y_true[:, 0] * 100, y_pred[:, 0] * 100, s=6, alpha=.4)
    ax[1, 0].plot([0, 100], [0, 100], "r--")
    ax[1, 0].set_title("Degradation speed %: true vs pred")
    ax[1, 0].set_xlabel("true"); ax[1, 0].set_ylabel("pred")

    err = (y_pred - y_true).ravel() * 100
    ax[1, 1].hist(err, bins=40)
    ax[1, 1].set_title(f"Risk error distribution (MAE={np.abs(err).mean():.2f}%)")

    n_cls = c_pred.shape[1]
    cm = np.zeros((n_cls, n_cls), int)
    pc = c_pred.argmax(1)
    for t, p in zip(c_true, pc):
        cm[t, p] += 1
    im = ax[1, 2].imshow(cm, cmap="Blues")
    ax[1, 2].set_title("Destructive-factor confusion")
    ax[1, 2].set_xticks(range(n_cls)); ax[1, 2].set_yticks(range(n_cls))
    ax[1, 2].set_xticklabels(CONFIG.FACTORS, rotation=45, ha="right")
    ax[1, 2].set_yticklabels(CONFIG.FACTORS)
    for i in range(n_cls):
        for j in range(n_cls):
            ax[1, 2].text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax[1, 2])

    fig.tight_layout()
    out = os.path.join(root, "dashboard.png")
    fig.savefig(out, dpi=130)
    print(f"[dashboard] saved -> {out}")


def estimate_workload(n_train):
    steps = math.ceil(n_train / CONFIG.BATCH_SIZE)
    print(f"\n[workload] steps/epoch = {steps}  (N_train={n_train}, batch={CONFIG.BATCH_SIZE})")
    print("[workload] TARGET ~3-5 h, hard cap < 6 h. Time the first epoch, then "
          "estimate: runtime ~= sec_per_epoch x epochs_to_converge.")
    print("[workload] too SLOW? raise BATCH_SIZE, raise STRIDE, lower VOXEL_RES or "
          "N_ARTIFACTS. Too FAST and under-trained? do the opposite. EarlyStopping "
          "stops at convergence; TimeBudgetStop caps the wall clock.")

 
def main():
    root = resolve_root()
    print(f"[init] output root: {root}")
    print(f"[init] GPUs: {tf.config.list_physical_devices('GPU')}")

    banks, samples, arts = build_dataset(root)
    samples = correct_imbalance(samples, target_ratio=0.20, seed=SEED)

    n = len(samples["win_id"])
    idx = np.arange(n)
    tr, tmp = train_test_split(idx, test_size=CONFIG.VAL_SPLIT + CONFIG.TEST_SPLIT,
                               random_state=SEED, stratify=samples["y_cls"])
    rel = CONFIG.TEST_SPLIT / (CONFIG.VAL_SPLIT + CONFIG.TEST_SPLIT)
    va, te = train_test_split(tmp, test_size=rel, random_state=SEED,
                              stratify=samples["y_cls"][tmp])
    print(f"[split] train={len(tr)} val={len(va)} test={len(te)}")
    estimate_workload(len(tr))

    model = build_model(n_classes=len(CONFIG.FACTORS))
    model.summary()

    ds_tr = make_tfds(tr, banks, samples, training=True)
    ds_va = make_tfds(va, banks, samples, training=False)
    ds_te = make_tfds(te, banks, samples, training=False)

    print("\n[train] starting (EarlyStopping=convergence, TimeBudgetStop=cap) ...")
    hist = model.fit(ds_tr, validation_data=ds_va, epochs=CONFIG.EPOCHS,
                     callbacks=make_callbacks(root), verbose=1)

    print("\n[eval] test set ...")
    metrics = model.evaluate(ds_te, return_dict=True, verbose=1)
    print(json.dumps({k: float(v) for k, v in metrics.items()}, indent=2))

    preds = model.predict(ds_te, verbose=0)
    eval_arrays = (samples["y_reg"][te], preds["risk"],
                   samples["y_cls"][te], preds["factor"])
    dashboard(hist, eval_arrays, root)

    model.save(os.path.join(root, "final_model.keras"))
    with open(os.path.join(root, "test_metrics.json"), "w") as f:
        json.dump({k: float(v) for k, v in metrics.items()}, f, indent=2)
    print(f"\n[done] artifacts written under {root}")


if __name__ == "__main__":
    main()


[init] output root: /content/drive/MyDrive/artifact_risk
[init] GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[climate] no CSV found; fetching real Open-Meteo archive ...
[climate] REAL archive loaded: 1460 days -> /content/drive/MyDrive/artifact_risk/climate_real.pkl
[meta] artifacts.json not found -> deterministic catalogue (replace with your real inspection records).
[voxel] building 300 voxel grids @ 20^3 (once each) ...
[assemble] 300 artifacts x 350 windows = 105000 samples (banks: env 0.8MB, voxel 9.6MB)
[assemble] total samples: 105000, raw hazard share: 5.43%
[imbalance] +19125 hazard duplicates -> hazard share = 20.0000% (target 20%)
[split] train=86887 val=18619 test=18619

[workload] steps/epoch = 679  (N_train=86887, batch=128)
[workload] TARGET ~3-5 h, hard cap < 6 h. Time the first epoch, then estimate: runtime ~= sec_per_epoch x epochs_to_converge.
[workload] too SLOW? raise BATCH_SIZE, raise STRIDE, lower VOXEL_RES or N_ARTIFACTS. Too FAST an

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ env (InputLayer)    │ (None, 60, 9)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 60, 50)    │      9,150 │ env[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ material            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ storage             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ voxel (InputLayer)  │ (None, 20, 20,    │          0 │ -                 │
│                     │ 20, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 60, 50)    │          0 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup       │ (None, 1)         │          0 │ material[0][0]    │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ string_lookup_1     │ (None, 1)         │          0 │ storage[0][0]     │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d (Conv3D)     │ (None, 20, 20,    │        448 │ voxel[0][0]       │
│                     │ 20, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 50)    │     40,650 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │         72 │ string_lookup[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 8)      │         48 │ string_lookup_1[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d       │ (None, 10, 10,    │          0 │ conv3d[0][0]      │
│ (MaxPooling3D)      │ 10, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 60, 50)    │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meta_num            │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_1 (Conv3D)   │ (None, 10, 10,    │     13,856 │ max_pooling3d[0]… │
│                     │ 10, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 356,715 (1.36 MB)

 Trainable params: 356,715 (1.36 MB)

 Non-trainable params: 0 (0.00 B)


[train] starting (EarlyStopping=convergence, TimeBudgetStop=cap) ...
Epoch 1/80
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - factor_acc: 0.7883 - factor_loss: 0.5010 - loss: 0.2684 - risk_loss: 0.0178 - risk_mae: 0.1053 - risk_mape: 23.2399
Epoch 1: val_loss improved from None to 0.05208, saving model to /content/drive/MyDrive/artifact_risk/checkpoints/best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/artifact_risk/checkpoints/best.weights.h5
   [time] elapsed 0.02 h / cap 6.0 h
679/679 ━━━━━━━━━━━━━━━━━━━━ 61s 69ms/step - factor_acc: 0.8937 - factor_loss: 0.2607 - loss: 0.1424 - risk_loss: 0.0121 - risk_mae: 0.0857 - risk_mape: 19.1973 - val_factor_acc: 0.9570 - val_factor_loss: 0.0962 - val_loss: 0.0521 - val_risk_loss: 0.0040 - val_risk_mae: 0.0509 - val_risk_mape: 11.4992 - learning_rate: 3.0000e-04
Epoch 2/80
679/679 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - factor_acc: 0.9650 - factor_loss: 0.0884 - loss: 0.0500 - risk_loss: 0.0058 - risk_mae: 0.0603 - risk_mape